# BÖLGE BAZLI RANDOM FOREST

Eski kurulumun iki temel sorunu vardı:
- **Veri azlığı:** tüm depremlerin haftalık/yıllık ortalaması alınınca 75 bin kayıt 42 satıra iniyordu.
- **Ortalamanın sinyali yok etmesi:** küresel ortalama büyüklük neredeyse sabit, ortalama enlem/boylam ise gerçek bir yer değil.

Burada dünya ızgara hücrelerine bölünüyor ve her hücre ayrı bir zaman serisi oluyor (ayrıntı: `regional.py`).
- **Satır:** (hücre, dönem)
- **Hedef:** aynı hücrede bir sonraki dönemin **en büyük** depremi (o dönemde en az bir deprem kaydı varsa)
- **Özellikler** (hepsi sadece şimdiki ve geçmiş dönemlerden):
  - son dönemlerin deprem sayısı, ortalama / en büyük büyüklük, büyüklük dağılımı, derinlik, büyük deprem sayısı, sismik enerji
  - hücrenin o ana kadarki geçmiş profili
  - son 4 ve 8 dönemin kayan pencere özetleri
  - komşu 8 hücrenin şimdiki aktivitesi
  - hücrenin koordinatı

Model üç basit referansla karşılaştırılıyor: persistence (gelecek = şimdiki), global ortalama ve hücre ortalaması (her hücrenin train'deki ortalaması).

In [1]:
import numpy as np
import pandas as pd
import joblib
import warnings
warnings.filterwarnings('ignore')
from sklearn.ensemble import RandomForestRegressor
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV
from regional import (build_panel, make_features, feature_columns, split_by_period, PeriodSplit,
                      LagSelector, evaluate_regional, TARGET_COL)

C:\Users\eren1\anaconda3\Lib\site-packages\pandas\core\computation\expressions.py:23: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.10.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED


In [2]:
def run_rf(df, name, cell_deg, freq, min_active, big_mag, max_lags, lag_options, model_path):
    panel = build_panel(df, cell_deg=cell_deg, freq=freq, min_active=min_active, big_mag=big_mag)
    data = make_features(panel, max_lags=max_lags)
    train, test = split_by_period(data)   # kronolojik: ilk %80 dönem train, son %20 test
    features = feature_columns(train)
    print(f"{name}: {panel['period'].nunique()} dönem, {len(features)} özellik")

    search = GridSearchCV(
        Pipeline([('lags', LagSelector()), ('model', RandomForestRegressor(n_estimators=300, random_state=42, n_jobs=1))]),
        {'lags__n_lags': lag_options,
         'model__max_depth': [12, None],
         'model__min_samples_leaf': [3, 10, 20],
         'model__max_features': [0.3, 0.5, 0.8]},
        cv=PeriodSplit(n_splits=3),   # validation her zaman train'den sonraki dönemler
        scoring='neg_mean_squared_error', error_score='raise', n_jobs=-1)
    search.fit(train[features], train[TARGET_COL], groups=train['period_idx'])
    print(f"En iyi parametreler: {search.best_params_}, CV MSE: {-search.best_score_:.4f}")

    regression, classification = evaluate_regional(
        'Random Forest', search.predict(train[features]), search.predict(test[features]), train, test, name)

    best = search.best_estimator_
    importance = pd.Series(best.named_steps['model'].feature_importances_, index=best.named_steps['lags'].columns_)
    print("\nEn önemli 10 özellik:")
    print(importance.sort_values(ascending=False).head(10).round(3).to_string())
    joblib.dump(best, model_path)
    return search, regression, classification

## DATASET 1 (USGS, 2022): 1°×1° hücre × hafta
Haftaların en az %5'inde deprem kaydı olan bütün hücreler dahil. Sadece en aktif hücrelerle sınırlı kalmak, modelin farklı türde bölgeleri (yoğun ağla izlenen küçük depremli bölgeler, sadece büyük depremlerin kaydedildiği okyanus bölgeleri) ayırt etmeyi öğrenmesini engelliyordu.

In [3]:
df1 = pd.read_csv('usgs_main.csv')
df1['time'] = pd.to_datetime(df1['time'])
df1 = df1[df1['type'] == 'earthquake']   # taş ocağı patlaması vb. kayıtlar deprem değil
search_1, regression_1, classification_1 = run_rf(
    df1, 'Bolgesel RF Dataset1', cell_deg=1.0, freq='W', min_active=0.05, big_mag=3.0,
    max_lags=8, lag_options=[4, 8], model_path='models/regional_rf_dataset1.pkl')

Bolgesel RF Dataset1: 40 dönem, 73 özellik


En iyi parametreler: {'lags__n_lags': 4, 'model__max_depth': 12, 'model__max_features': 0.3, 'model__min_samples_leaf': 20}, CV MSE: 0.3911



=== Bolgesel RF Dataset1 | 1164 hücre, train: 6160 satır, test: 1884 satır ===

Regresyon metrikleri (hedef: bir sonraki dönemin en büyük depremi)
                                      R²    MAE    MSE   RMSE
Model                       Veri                             
Random Forest               Train  0.834  0.398  0.309  0.556
                            Test   0.817  0.427  0.355  0.596
Persistence (referans)      Train  0.627  0.593  0.694  0.833
Global ortalama (referans)  Train  0.000  1.158  1.861  1.364
Hücre ortalaması (referans) Train  0.832  0.391  0.313  0.560
Persistence (referans)      Test   0.694  0.563  0.594  0.771
Global ortalama (referans)  Test  -0.041  1.192  2.023  1.422
Hücre ortalaması (referans) Test   0.632  0.583  0.714  0.845

Sınıflandırma metrikleri (eşikler: train hedef dağılımının %75 ve %90'lık dilimleri)
                                               Pozitif oranı  Accuracy  Precision  Recall     F1    AUC
Model                       Veri  Eşik    

## DATASET 2 (M≥5.5, küresel): 10°×10° hücre × yıl
1973 sonrası kullanılıyor; öncesinde yıllık kayıt sayısı çok daha az olduğu için dönemler karşılaştırılabilir değil. Yılların en az %20'sinde kaydı olan hücreler dahil.

In [4]:
df2 = pd.read_csv('Significant Earthquake Dataset 1900-2023.csv')
df2 = df2.rename(columns={'Time':'time','Mag':'mag','Depth':'depth','Latitude':'latitude','Longitude':'longitude'})
df2['time'] = pd.to_datetime(df2['time'])
df2 = df2[df2['time'].dt.year >= 1973]
search_2, regression_2, classification_2 = run_rf(
    df2, 'Bolgesel RF Dataset2', cell_deg=10.0, freq='Y', min_active=0.2, big_mag=6.5,
    max_lags=6, lag_options=[3, 6], model_path='models/regional_rf_dataset2.pkl')

Bolgesel RF Dataset2: 49 dönem, 56 özellik


En iyi parametreler: {'lags__n_lags': 6, 'model__max_depth': None, 'model__max_features': 0.3, 'model__min_samples_leaf': 20}, CV MSE: 0.2460



=== Bolgesel RF Dataset2 | 161 hücre, train: 3232 satır, test: 905 satır ===

Regresyon metrikleri (hedef: bir sonraki dönemin en büyük depremi)
                                      R²    MAE    MSE   RMSE
Model                       Veri                             
Random Forest               Train  0.443  0.329  0.185  0.430
                            Test   0.252  0.381  0.249  0.499
Persistence (referans)      Train -0.327  0.494  0.441  0.664
Global ortalama (referans)  Train  0.000  0.460  0.332  0.577
Hücre ortalaması (referans) Train  0.330  0.362  0.223  0.472
Persistence (referans)      Test  -0.403  0.501  0.467  0.684
Global ortalama (referans)  Test  -0.002  0.460  0.334  0.578
Hücre ortalaması (referans) Test   0.236  0.377  0.254  0.504

Sınıflandırma metrikleri (eşikler: train hedef dağılımının %75 ve %90'lık dilimleri)
                                               Pozitif oranı  Accuracy  Precision  Recall     F1    AUC
Model                       Veri  Eşik      